# Time-Aware Hyperparameter Tuning — Final Forecast Model

This notebook tunes the forecasting model selected after the feature-ablation benchmark without contaminating the final out-of-sample evaluation.

The key principle is **nested temporal validation**: the outer forecast year remains untouched, while hyperparameters are selected using only earlier years through an expanding-window inner validation scheme.

## 1. Validation design

For each outer forecast year `T`:

1. Training data contain only years `< T`.
2. Within that training period, candidate hyperparameters are evaluated using expanding-window inner folds.
3. The best configuration is selected using mean inner-fold MAE.
4. The selected model is refit on **all** observations before `T`.
5. It is evaluated once on year `T`.

This prevents information from the outer test year from influencing model selection.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'data').exists(): PROJECT_ROOT=PROJECT_ROOT.parent
FEATURE_PATH=PROJECT_ROOT/'data'/'processed'/'forecasting_features.csv'
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f'{FEATURE_PATH} not found. Execute Notebook 09 first.')

df=pd.read_csv(FEATURE_PATH)
df['year']=pd.to_numeric(df['year'],errors='coerce')
df['municipality_code']=df['municipality_code'].astype('string')
df=df.dropna(subset=['year','target_price_growth']).copy()

advanced_features=[
    'price_growth','ntn_growth','population_growth','ntn_per_1000',
    'price_growth_lag1','price_growth_lag2','ntn_growth_lag1','ntn_growth_lag2',
    'population_growth_lag1','population_growth_lag2',
    'price_momentum_1y','ntn_momentum_1y','population_momentum_1y',
    'price_growth_roll2_mean','price_growth_roll3_mean','price_growth_roll3_std',
    'ntn_growth_roll2_mean','ntn_growth_roll3_mean','ntn_growth_roll3_std',
    'population_growth_roll2_mean','population_growth_roll3_mean','population_growth_roll3_std',
    'ntn_per_1000_lag1','ntn_per_1000_lag2',
    'log_price_lag1','log_ntn_lag1','log_population_lag1',
    'region_price_growth_lag1','region_ntn_growth_lag1','region_population_growth_lag1',
    'municipality_minus_region_price_growth_lag1','municipality_minus_region_ntn_growth_lag1'
]
missing=[c for c in advanced_features if c not in df.columns]
if missing: raise KeyError(f'Missing expected advanced features: {missing}')

years=sorted(df['year'].astype(int).unique())
print(f'Rows: {len(df):,} | Years: {years}')

## 2. Candidate hyperparameters

The grid is intentionally compact because the historical time dimension is limited. It varies the main complexity controls of `HistGradientBoostingRegressor`; the evaluation metric remains MAE.

In [ ]:
param_grid=[
    {'learning_rate':0.03,'max_iter':150,'max_leaf_nodes':7,'l2_regularization':1.0},
    {'learning_rate':0.03,'max_iter':250,'max_leaf_nodes':7,'l2_regularization':1.0},
    {'learning_rate':0.05,'max_iter':150,'max_leaf_nodes':15,'l2_regularization':1.0},
    {'learning_rate':0.05,'max_iter':250,'max_leaf_nodes':15,'l2_regularization':1.0},
    {'learning_rate':0.05,'max_iter':200,'max_leaf_nodes':31,'l2_regularization':1.0},
    {'learning_rate':0.05,'max_iter':200,'max_leaf_nodes':15,'l2_regularization':5.0},
    {'learning_rate':0.10,'max_iter':150,'max_leaf_nodes':15,'l2_regularization':5.0},
    {'learning_rate':0.03,'max_iter':250,'max_leaf_nodes':15,'l2_regularization':5.0},
]
len(param_grid)

## 3. Model factory

Median imputation is fitted inside each training fold. The estimator therefore never learns missing-value statistics from the validation or outer test data.

In [ ]:
def make_model(params):
    return Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('model',HistGradientBoostingRegressor(
            **params, random_state=42
        )),
    ])

def inner_folds(train_years):
    train_years=sorted(train_years)
    # Each validation year is predicted using all preceding years.
    return [(train_years[:i],train_years[i]) for i in range(2,len(train_years))]

## 4. Nested rolling-origin tuning

The first eligible outer years are skipped when there are not enough earlier years to create meaningful inner folds. This is preferable to introducing random cross-validation into a time-series problem.

In [ ]:
outer_results=[]
tuning_log=[]

for outer_year in years:
    outer_train=df[df['year']<outer_year].copy()
    outer_test=df[df['year']==outer_year].copy()
    train_years=sorted(outer_train['year'].astype(int).unique())
    folds=inner_folds(train_years)
    if len(folds)<2 or outer_test.empty:
        continue

    scores=[]
    for candidate_id,params in enumerate(param_grid):
        fold_scores=[]
        for fit_years,val_year in folds:
            inner_train=outer_train[outer_train['year'].isin(fit_years)]
            inner_val=outer_train[outer_train['year'].eq(val_year)]
            model=make_model(params)
            model.fit(inner_train[advanced_features],inner_train['target_price_growth'])
            pred=model.predict(inner_val[advanced_features])
            fold_scores.append(mean_absolute_error(inner_val['target_price_growth'],pred))
        scores.append({'candidate_id':candidate_id,'mean_inner_mae':np.mean(fold_scores),'median_inner_mae':np.median(fold_scores),'params':params})

    tuning_table=pd.DataFrame(scores).sort_values(['mean_inner_mae','median_inner_mae']).reset_index(drop=True)
    best=tuning_table.iloc[0]
    best_params=best['params']
    final_model=make_model(best_params)
    final_model.fit(outer_train[advanced_features],outer_train['target_price_growth'])
    pred=final_model.predict(outer_test[advanced_features])

    outer_results.append({
        'forecast_year':outer_year,
        'selected_candidate':int(best['candidate_id']),
        'inner_mae':float(best['mean_inner_mae']),
        'outer_mae':mean_absolute_error(outer_test['target_price_growth'],pred),
        'outer_rmse':np.sqrt(mean_squared_error(outer_test['target_price_growth'],pred)),
        'outer_r2':r2_score(outer_test['target_price_growth'],pred),
        'n_train':len(outer_train),
        'n_test':len(outer_test),
    })
    tuning_log.append({'forecast_year':outer_year,'selected_candidate':int(best['candidate_id']),'params':best_params,'inner_mae':float(best['mean_inner_mae'])})

outer_results_df=pd.DataFrame(outer_results)
print(f'Completed {len(outer_results_df)} nested outer folds.')
display(outer_results_df)

## 5. Final out-of-sample performance

These metrics are based exclusively on outer forecast years. They are the appropriate figures to compare with the fixed-parameter benchmark in Notebook 10.

In [ ]:
final_summary=pd.DataFrame({
    'metric':['Mean MAE','Median MAE','Mean RMSE','Mean R²'],
    'value':[
        outer_results_df['outer_mae'].mean(),
        outer_results_df['outer_mae'].median(),
        outer_results_df['outer_rmse'].mean(),
        outer_results_df['outer_r2'].mean(),
    ]
})
display(final_summary)

## 6. Hyperparameter stability

A configuration repeatedly selected across outer folds is generally preferable to one selected only once. This section summarizes which candidate configurations were chosen.

In [ ]:
selection_counts=outer_results_df['selected_candidate'].value_counts().sort_index().rename('selected_outer_folds')
display(selection_counts.to_frame())

for candidate_id,count in selection_counts.items():
    print(f'Candidate {candidate_id}: selected in {int(count)} outer folds -> {param_grid[int(candidate_id)]}')

## 7. Interpretation

The final model should be reported only after checking three things:

1. nested outer-fold MAE is competitive with the fixed benchmark;
2. gains are not concentrated in a single forecast year;
3. selected hyperparameters are reasonably stable across outer folds.

Because the dataset contains relatively few annual periods, tuning a large grid would increase model-selection variance. The compact grid is therefore intentional.

This remains a predictive exercise: the model does not establish that any feature causes property-price growth. OMI quotations, spatial dependence and omitted macroeconomic factors remain important limitations.

## 8. Portfolio-ready conclusion

The appropriate final statement is based on the observed outer-fold results rather than the best individual fold. If nested tuning does not improve the fixed benchmark materially, the simpler fixed specification should be preferred because it has lower model-selection complexity.

If the tuned model produces a persistent out-of-sample gain, it can be presented as the final forecasting specification, with the validation protocol documented explicitly.